In [1]:
from pathlib import Path
import pandas as pd

In [2]:
# load the prepared datasets
project_dir = Path.cwd().parent.parent
interim_dir = project_dir / "interim"

data = {
    year: pd.read_csv(interim_dir / f"opencity_electiondata_{year}.csv")
    for year in [2014, 2019, 2024]
}

In [3]:
# filter datasets to selected states

df_2014 = data[2014][
    data[2014]["State_Name"].isin(
        ["Bihar", "Maharashtra", "Uttar_Pradesh", "Tamil_Nadu"]
    )
].copy()

df_2019 = data[2019][
    data[2019]["State_Name"].isin(
        ["Bihar", "Maharashtra", "Uttar_Pradesh", "Tamil_Nadu"]
    )
].copy()

df_2024 = data[2024][
    data[2024]["State"].isin(
        ["Bihar", "Maharashtra", "Uttar Pradesh", "Tamil Nadu"]
    )
].copy()

In [4]:
# normalize column and constituency names

df_2014 = df_2014.rename(columns={"State_Name": "State"})
df_2019 = df_2019.rename(columns={"State_Name": "State"})

df_2024 = df_2024.rename(columns={
    "PC No": "Constituency_No",
    "PC Name": "Constituency_Name"
})

for df in [df_2014, df_2019]:
    df["State"] = df["State"].replace({
        "Uttar_Pradesh": "Uttar Pradesh",
        "Tamil_Nadu": "Tamil Nadu"
    })

for df in [df_2014, df_2019, df_2024]:
    df["Constituency_Name"] = (
        df["Constituency_Name"].str.strip().str.upper()
    )

In [5]:
# check vote consistency assuming '-' means zero
votes = df_2024[["EVM Votes", "Postal Votes", "Total Votes"]].replace("-", "0")
votes = votes.apply(pd.to_numeric, errors="coerce")

complete = votes.notna().all(axis=1)
matches = (
    votes["EVM Votes"] + votes["Postal Votes"] == votes["Total Votes"]
)

print("Rows checked:", complete.sum())
print("Rows where EVM + Postal = Total:", (complete & matches).sum())
print("Rows where EVM + Postal != Total:", (complete & ~matches).sum())
print("Rows with missing or invalid vote values:", (~complete).sum())

Rows checked: 3626
Rows where EVM + Postal = Total: 3626
Rows where EVM + Postal != Total: 0
Rows with missing or invalid vote values: 0


In [6]:
# keep only columns needed for analysis

cols_2014_2019 = [
    "State", "Constituency_No", "Constituency_Name",
    "Year", "Candidate", "Party", "Position", "Votes",
    "Vote_Share_Percentage", "Margin"
]

cols_2024 = [
    "State", "Constituency_No", "Constituency_Name",
    "Candidate", "Party", "Total Votes", "Vote Share"
]

df_2014 = df_2014[cols_2014_2019].copy()
df_2019 = df_2019[cols_2014_2019].copy()
df_2024 = df_2024[cols_2024].copy()

### Purpose of the Kept Columns: 2014 and 2019

- **State:** Identify the state in which the candidate contested and support state-wise analysis.
- **Constituency_No, Constituency_Name:** Identify parliamentary constituencies and support matching across election years.
- **Year:** Identify the election year for comparing candidate performance across 2014 and 2019.
- **Candidate:** Match candidates with their MyNeta asset records.
- **Party:** Provide additional context for candidate identification and track party affiliation.
- **Position:** Identify the candidate's finishing position and determine whether they won or lost.
- **Votes:** Compare candidates' vote totals and assess their electoral performance.
- **Vote_Share_Percentage:** Compare candidates' shares of the valid votes cast, even when constituencies differ in size.
- **Margin:** Measure the difference between the winner's and runner-up's vote totals, providing an indication of how competitive the constituency was.

### Purpose of the Kept Columns: 2024

- **State:** Identify the state in which the candidate contested.
- **Constituency_No, Constituency_Name:** Identify the parliamentary constituency and support matching across datasets.
- **Candidate:** Match candidates with their MyNeta asset records.
- **Party:** Provide additional context for candidate identification.
- **Total Votes:** Compare candidates' vote totals and help identify the election winner.

`EVM Votes` and `Postal Votes` are excluded because all 8,902 rows satisfy `EVM Votes + Postal Votes = Total Votes` when `-` is treated as zero. This supports treating `-` as zero for this consistency check, while retaining `Total Votes` for analysing election outcomes.

The 2024 dataset does not provide the same `Position`, `Vote_Share_Percentage`, and `Margin` columns as the 2014 and 2019 datasets. These can be derived from the 2024 vote totals where needed, allowing the datasets to be compared using a consistent set of measures.

In [7]:
# check vote percentages are between 0 and 100

for year, df, column in [
    (2014, df_2014, "Vote_Share_Percentage"),
    (2019, df_2019, "Vote_Share_Percentage"),
    (2024, df_2024, "Vote Share")
]:
    values = pd.to_numeric(df[column], errors="coerce")

    print(f"\n{'=' * 8} {year} {'=' * 8}")
    print("Total rows:", len(df))
    print("Missing or invalid:", values.isna().sum())
    print("Below 0:", (values < 0).sum())
    print("Above 100:", (values > 100).sum())


======== 2014 ========
Total rows: 3868
Missing or invalid: 0
Below 0: 0
Above 100: 0

======== 2019 ========
Total rows: 3545
Missing or invalid: 0
Below 0: 0
Above 100: 0

======== 2024 ========
Total rows: 3626
Missing or invalid: 0
Below 0: 0
Above 100: 0


In [8]:
# inspect vote-share data types

for year, df, column in [
    (2014, df_2014, "Vote_Share_Percentage"),
    (2019, df_2019, "Vote_Share_Percentage"),
    (2024, df_2024, "Vote Share")
]:
    print(f"\n{'=' * 8} {year} {'=' * 8}")
    print("Original dtype:", df[column].dtype)
    print("Sample values:")
    print(df[column].head().to_list())


======== 2014 ========
Original dtype: float64
Sample values:
[40.44, 27.35, 9.07, 6.53, 3.99]

======== 2019 ========
Original dtype: float64
Sample values:
[58.39, 24.03, 6.1, 3.33, 1.44]

======== 2024 ========
Original dtype: object
Sample values:
['47.5', '38.55', '2.11', '1.8', '1.71']


In [9]:
# convert 2024 vote share to numeric

df_2024["Vote Share"] = pd.to_numeric(df_2024["Vote Share"])

In [10]:
# check vote-share totals by constituency

for year, df, column in [
    (2014, df_2014, "Vote_Share_Percentage"),
    (2019, df_2019, "Vote_Share_Percentage"),
    (2024, df_2024, "Vote Share")
]:
    totals = df.groupby(
        ["State", "Constituency_No", "Constituency_Name"]
    )[column].sum()

    unusual = totals[(totals - 100).abs() > 0.5]

    print(f"\n{'=' * 8} {year} {'=' * 8}")
    print("Constituencies checked:", len(totals))
    print("Unusual totals:", len(unusual))
    print(unusual.sort_values(ascending=False).to_string())


======== 2014 ========
Constituencies checked: 208
Unusual totals: 1
State        Constituency_No  Constituency_Name
Maharashtra  39               BEED                 200.0

======== 2019 ========
Constituencies checked: 208
Unusual totals: 1
State        Constituency_No  Constituency_Name
Maharashtra  45               SATARA               200.0

======== 2024 ========
Constituencies checked: 207
Unusual totals: 0
Series([], )


In [11]:
# drop constituencies with combined election records

df_2014 = df_2014[
    ~((df_2014["State"] == "Maharashtra") &
      (df_2014["Constituency_Name"] == "BEED"))
].copy()

df_2019 = df_2019[
    ~((df_2019["State"] == "Maharashtra") &
      (df_2019["Constituency_Name"] == "SATARA"))
].copy()

In [12]:
# verify remaining datasets

for year, df in [(2014, df_2014), (2019, df_2019), (2024, df_2024)]:
    print(year, "rows:", len(df))

2014 rows: 3816
2019 rows: 3527
2024 rows: 3626


In [13]:
# derive comparable columns for 2024

df_2024["Votes"] = pd.to_numeric(df_2024["Total Votes"])
df_2024["Vote_Share_Percentage"] = pd.to_numeric(df_2024["Vote Share"])

In [14]:
# rank candidates within each constituency

groups = ["State", "Constituency_No", "Constituency_Name"]

df_2024["Position"] = (
    df_2024.groupby(groups)["Votes"]
    .rank(method="min", ascending=False)
    .astype(int)
)

In [15]:
# sort candidates within each constituency by votes

df_2024 = df_2024.sort_values(
    groups + ["Votes"],
    ascending=[True, True, True, False]
).copy()

In [16]:
# calculate margin over the next candidate

df_2024["Margin"] = (
    df_2024["Votes"]
    - df_2024.groupby(groups)["Votes"].shift(-1)
)

In [17]:
# add election year

df_2024["Year"] = 2024

In [18]:
# standardize columns across election years

cols = [
    "State", "Constituency_No", "Constituency_Name",
    "Year", "Candidate", "Party", "Position", "Votes",
    "Vote_Share_Percentage", "Margin"
]

df_2014 = df_2014[cols].copy()
df_2019 = df_2019[cols].copy()
df_2024 = df_2024[cols].copy()

In [19]:
# compare columns across datasets

for year, df in [(2014, df_2014), (2019, df_2019), (2024, df_2024)]:
    print(f"\n{'=' * 8} {year} {'=' * 8}")
    print(df.columns.tolist())


======== 2014 ========
['State', 'Constituency_No', 'Constituency_Name', 'Year', 'Candidate', 'Party', 'Position', 'Votes', 'Vote_Share_Percentage', 'Margin']

======== 2019 ========
['State', 'Constituency_No', 'Constituency_Name', 'Year', 'Candidate', 'Party', 'Position', 'Votes', 'Vote_Share_Percentage', 'Margin']

======== 2024 ========
['State', 'Constituency_No', 'Constituency_Name', 'Year', 'Candidate', 'Party', 'Position', 'Votes', 'Vote_Share_Percentage', 'Margin']


In [20]:
# final sanity checks

for year, df in [(2014, df_2014), (2019, df_2019), (2024, df_2024)]:
    groups = ["State", "Constituency_No", "Constituency_Name"]

    expected_position = df.groupby(groups)["Votes"].rank(
        method="min", ascending=False
    )

    ordered = df.sort_values(
        groups + ["Votes"],
        ascending=[True, True, True, False]
    )

    expected_margin = (
        ordered.groupby(groups)["Votes"]
        .transform(lambda votes: votes - votes.shift(-1))
    )

    print(f"\n{'=' * 8} {year} {'=' * 8}")
    print("Rows:", len(df))
    print("Duplicate rows:", df.duplicated().sum())
    print("Missing values:\n", df.isna().sum().to_string())
    print("Position mismatches:", (df["Position"] != expected_position).sum())
    print("Negative votes:", (df["Votes"] < 0).sum())
    print("Invalid vote shares:", (
        df["Vote_Share_Percentage"].isna()
        | (df["Vote_Share_Percentage"] < 0)
        | (df["Vote_Share_Percentage"] > 100)
    ).sum())
    print("Negative margins:", (df["Margin"] < 0).sum())
    print("Margin mismatches:", (
        ~(
            ordered["Margin"].eq(expected_margin)
            | (ordered["Margin"].isna() & expected_margin.isna())
        )
    ).sum())


======== 2014 ========
Rows: 3816
Duplicate rows: 0
Missing values:
 State                    0
Constituency_No          0
Constituency_Name        0
Year                     0
Candidate                0
Party                    0
Position                 0
Votes                    0
Vote_Share_Percentage    0
Margin                   0
Position mismatches: 8
Negative votes: 0
Invalid vote shares: 0
Negative margins: 0
Margin mismatches: 207

======== 2019 ========
Rows: 3527
Duplicate rows: 0
Missing values:
 State                    0
Constituency_No          0
Constituency_Name        0
Year                     0
Candidate                0
Party                    1
Position                 0
Votes                    0
Vote_Share_Percentage    0
Margin                   0
Position mismatches: 4
Negative votes: 0
Invalid vote shares: 0
Negative margins: 0
Margin mismatches: 207

======== 2024 ========
Rows: 3626
Duplicate rows: 0
Missing values:
 State                      0
Constit